# Documentação — 04 - Ingestão Bronze: Detecção e Carga Incremental de Clientes
**Squad 2 — Real Time for Business | Dupla 2**  
**Integrantes:** Kauan & Leandro  
**Tabelas de Escopo:** `ecommerce_clientes` (arquivos Parquet no ADLS Gen2)  
**Branch:** `feat/squad2/kauan-leandro`  

Este notebook faz a **ingestão incremental da camada Bronze** no ADLS Gen2. Ele detecta novos arquivos Parquet no container `raw`, carrega apenas os arquivos ainda não processados e grava na camada Bronze no ADLS.

**Funcionalidades principais:**
- Detecção automática de novos micro-lotes via log de controle
- Escrita em Delta Lake na camada Bronze
- Loop de monitoramento (15s de intervalo, 90s de duração)

**⚠️ Atenção:** Este notebook usa Azure SDK + ADLS, que **não funciona no serverless**. No serverless, a ingestão Bronze é feita via Lakehouse Federation (`sqlserver_catalog.squad2.ecommerce_clientes`).

## Célula 1 — Carregamento de credenciais

```python
%run ./00_setup_config.ipynb
```

**O que faz:** Executa o notebook de setup, carregando as credenciais do `.env` (ADLS e SQL Server). A partir daqui, `os.getenv("ADLS_CLIENT_ID")` etc. estão disponíveis.

## Célula 2 — Importação de bibliotecas

```python
from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient
from dotenv import load_dotenv
from pathlib import Path
from pyspark.sql import functions as F
import os
```

**O que faz:** Importa:
- **Azure SDK** (`ClientSecretCredential`, `DataLakeServiceClient`) — acesso ao ADLS Gen2
- **dotenv** (`load_dotenv`) — carrega credenciais do `.env`
- **pathlib** (`Path`) — busca dinâmica do `.env` no workspace
- **PySpark** (`F`) — funções para transformações
- **os** — leitura de variáveis de ambiente (`os.getenv`)

## Células 3–5 — Autenticação e conexão ADLS

```python
# Célula 3: Carrega .env
# Célula 4: Cria credencial Azure (ClientSecretCredential)
# Célula 5: Conecta ao ADLS (DataLakeServiceClient + container raw)
```

**O que faz:** Mesma sequência do notebook `01_conexao_adls_gen2`:
1. Carrega o `.env`
2. Cria a credencial de Service Principal
3. Conecta ao ADLS Gen2 e obtém o `fs_client` do container `raw`

## Célula 6 — Listagem de arquivos Parquet

```python
for arquivo in fs_client.get_paths(path="real-time-data", recursive=True):
    if arquivo.name.endswith("ecommerce_clientes.parquet"):
        arquivos_encontrados.append(arquivo.name)
```

**O que faz:** Lista todos os arquivos `ecommerce_clientes.parquet` no path `real-time-data/` do container `raw`. Estes são os micro-lotes gerados em tempo real pela empresa.

## Célula 7 — Configuração OAuth para Spark ABFS

```python
adls_options = {
    f"fs.azure.account.auth.type.{storage_account_name}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type....": "...ClientCredsTokenProvider",
    ...
}
```

**O que faz:** Configura as opções OAuth para o Spark acessar o ADLS via ABFS (Azure Blob File System). Estas opções são usadas pelo `spark.read.format("delta").options(**adls_options).load(caminho)` para ler e escrever Delta tables no ADLS.

## Célula 8 — Conexão via Azure SDK

```python
adls_tenant_id = os.getenv("ADLS_TENANT_ID")
adls_client_id = os.getenv("ADLS_CLIENT_ID")
adls_client_secret = os.getenv("ADLS_CLIENT_SECRET")
adls_storage_account = os.getenv("ADLS_STORAGE_ACCOUNT_NAME", "internshipdatalake")

import pandas as pd
from io import BytesIO

credential = ClientSecretCredential(adls_tenant_id, adls_client_id, adls_client_secret)
service_client = DataLakeServiceClient(...)
file_system_client = service_client.get_file_system_client(file_system="raw")
```

**O que faz:**
1. Lê credenciais do ADLS do `.env` (sobrescreve a variável `credential` da celula 4)
2. Importa **pandas** (`pd`) e **BytesIO** — usados para ler arquivos Parquet baixados
3. Cria nova conexão Azure SDK com `file_system_client` — usada para **listar e baixar arquivos** diretamente do ADLS

**Nota:** Segunda conexão além do Spark ABFS. `file_system_client` (aqui) = download de arquivos. `fs_client` (celula 5) + `adls_options` (celula 7) = leitura/escrita Delta via Spark.

## Célula 9 — Definição de caminhos Bronze

```python
caminho_bronze = f"abfss://squad2@{storage_account_name}.dfs.core.windows.net/bronze/ecommerce_clientes"
caminho_control_log = f"abfss://squad2@.../grupo2/metadata/ingestion_control_log"
```

**O que faz:** Define os caminhos no ADLS para:
- `caminho_bronze` — onde os dados da camada Bronze serão gravados (Delta Lake)
- `caminho_control_log` — onde o log de controle rastreia quais arquivos já foram processados

## Célula 10 — Função de verificação de tabela Delta

```python
def tabela_delta_existe(caminho):
    try:
        spark.read.format("delta").options(**adls_options).load(caminho).limit(1).count()
        return True
    except Exception:
        return False
```

**O que faz:** Verifica se um caminho Delta já existe no ADLS. Usado para decidir se é a primeira carga (cria a tabela) ou se é incremental (lê o log de controle).

## Célula 11 — Detecção de novos arquivos

```python
def detectar_novos_clientes():
    # 1. Lista todos os arquivos Parquet no ADLS
    todos_arquivos = [
        p.name for p in file_system_client.get_paths(path="real-time-data", recursive=True)
        if p.name.endswith("ecommerce_clientes.parquet")
    ]

    # 2. Lê o log de controle para ver quais já foram processados
    if tabela_delta_existe(caminho_control_log):
        df_control = spark.read.format("delta").options(**adls_options).load(caminho_control_log)
        ja_processados = {r["arquivo_processado"] for r in df_control.select("arquivo_processado").collect()}
    else:
        ja_processados = set()

    # 3. Filtra apenas arquivos novos
    arquivos_novos = [a for a in todos_arquivos if a not in ja_processados]
    if not arquivos_novos:
        return None, []

    # 4. Baixa cada arquivo via Azure SDK, lê com pandas e converte para Spark
    dfs = []
    for caminho in arquivos_novos:
        file_client = file_system_client.get_file_client(caminho)
        conteudo = file_client.download_file().readall()
        pdf = pd.read_parquet(BytesIO(conteudo))
        pdf["bronze_source_file"] = caminho  # rastreabilidade
        dfs.append(pdf)

    pdf_final = pd.concat(dfs, ignore_index=True)
    df_novos = spark.createDataFrame(pdf_final)
    return df_novos, arquivos_novos
```

**O que faz:** Esta é a **lógica incremental central**:
1. Lista todos os arquivos `ecommerce_clientes.parquet` no ADLS via `file_system_client`
2. Lê o log de controle (Delta) para ver quais arquivos já foram processados
3. Filtra apenas arquivos novos (que não estão no log)
4. **Baixa cada arquivo** via Azure SDK (`download_file().readall()`)
5. Lê com **pandas** (`pd.read_parquet(BytesIO(conteudo))`)
6. Adiciona coluna **`bronze_source_file`** — rastreia de qual arquivo cada registro veio
7. Concatena com `pd.concat` e converte para Spark DataFrame

Isto evita reprocessar arquivos já ingeridos na Bronze.

## Célula 12 — Schema do log de controle

```python
schema_log = StructType([
    StructField("tabela_origem", StringType(), False),
    StructField("arquivo_processado", StringType(), False),
    StructField("total_linhas", LongType(), False),
    StructField("status", StringType(), False),
    StructField("timestamp_processamento", StringType(), False),
])
```

**O que faz:** Define o schema da tabela de log de controle. Cada vez que um arquivo é processado, um registro é gravado com o nome do arquivo, total de linhas, status e timestamp. Isso garante rastreabilidade.

### Função `processar_bronze()` (mesma célula)

```python
def processar_bronze(df_novos, arquivos_novos):
    total = df_novos.count()

    # 1. Adiciona colunas de particionamento temporal
    df_bronze = (df_novos
        .withColumn("bronze_ingested_at", current_timestamp())
        .withColumn("ano", year(col("bronze_ingested_at")))
        .withColumn("mes", month(col("bronze_ingested_at")))
        .withColumn("dia", dayofmonth(col("bronze_ingested_at")))
        .withColumn("hora", hour(col("bronze_ingested_at"))))

    # 2. Grava na Bronze (Delta, append, particionado)
    (df_bronze.write.format("delta").options(**adls_options)
        .mode("append")
        .partitionBy("ano", "mes", "dia", "hora")
        .save(caminho_bronze))

    # 3. Grava log de controle por arquivo
    log_data = [("ecommerce_clientes", arquivo, linhas, "SUCCESS", agora) ...]
    df_log = spark.createDataFrame(log_data, schema=schema_log)
    df_log.write.format("delta").options(**adls_options).mode("append").save(caminho_control_log)
```

**O que faz a função `processar_bronze`:**
1. Adiciona **`bronze_ingested_at`** (timestamp de ingestão)
2. Adiciona colunas de **particionamento temporal**: `ano`, `mes`, `dia`, `hora`
3. Grava na Bronze no ADLS em formato **Delta**, modo **append**, particionado por data/hora
4. Cria um **log por arquivo** processado (nome, total de linhas, status, timestamp)
5. Grava o log no caminho `caminho_control_log` — usado pela função `detectar_novos_clientes()` para não reprocessar arquivos

## Célula 13 — Loop de monitoramento

```python
intervalo_segundos = 15
duracao_maxima_segundos = 90
MAX_FALHAS_CONSECUTIVAS = 5

while datetime.now() < fim_previsto:
    try:
        df_novos, arquivos_novos = detectar_novos_clientes()
        processar_bronze(df_novos, arquivos_novos)
        falhas_consecutivas = 0
    except Exception as e:
        falhas_consecutivas += 1
        if falhas_consecutivas >= MAX_FALHAS_CONSECUTIVAS:
            break
    time.sleep(intervalo_segundos)
```

**O que faz:** Executa um loop que:
1. A cada 15 segundos, chama `detectar_novos_clientes()` (verifica novos arquivos no ADLS)
2. Se houver novos, chama `processar_bronze()` (lê, grava na Bronze Delta com particionamento, atualiza log)
3. Se houver 5 falhas consecutivas, interrompe por segurança
4. Continua até atingir 90 segundos de duração máxima

**Este é o coração do pipeline em tempo real** — detecta e processa novos micro-lotes automaticamente.

## Resumo do fluxo

| Célula | Função |
| --- | --- |
| 1 | Carrega credenciais (%run 00_setup_config) |
| 2 | Importa 6 bibliotecas: Azure SDK, dotenv, pathlib, PySpark, os |
| 3 | Carrega `.env` via `rglob` (busca dinâmica) |
| 4 | Cria credencial Azure (`ClientSecretCredential`) |
| 5 | Conecta ao ADLS (`DataLakeServiceClient` + `fs_client`) |
| 6 | Lista arquivos `ecommerce_clientes.parquet` no ADLS |
| 7 | Configura OAuth para Spark ABFS (`adls_options`) |
| 8 | Segunda conexão Azure SDK (`file_system_client`) + importa pandas/BytesIO |
| 9 | Define caminhos Bronze e log no ADLS |
| 10 | Função `tabela_delta_existe()` — verifica se caminho Delta existe |
| 11 | Função `detectar_novos_clientes()` — baixa arquivos, lê com pandas, adiciona `bronze_source_file`, converte para Spark |
| 12 | Schema do log + função `processar_bronze()` — grava Bronze com particionamento (ano/mes/dia/hora) e atualiza log |
| 13 | Loop de monitoramento (15s, 90s, max 5 falhas) — chama `detectar_novos_clientes()` + `processar_bronze()` |

**Fluxo:** Autentica → lista → detecta novos → baixa via SDK → lê com pandas → converte para Spark → grava Bronze (Delta, particionado) → registra log → repete

**⚠️ Serverless:** Não funciona no serverless (Azure SDK). No serverless, a Bronze é acessada via `sqlserver_catalog.squad2.ecommerce_clientes` (Lakehouse Federation).